# Topic Modelling Starter

This notebook sets up a reusable workflow for LDA and BERTopic on the news dataset.

In [13]:
import re
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from bertopic import BERTopic

from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from sklearn.decomposition import LatentDirichletAllocation
from sklearn.feature_extraction.text import CountVectorizer



In [14]:
import nltk

nltk.download("punkt")
nltk.download("stopwords")

[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\WEWL\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\WEWL\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

In [15]:
DATA_PATH = Path("Data/NLP_data.csv")
TEXT_COLUMN = "content"
SAMPLE_SIZE = 5000

df = pd.read_csv(DATA_PATH)
df = df.dropna(subset=[TEXT_COLUMN]).copy()
df = df[["article_id", "title", "category", TEXT_COLUMN]].copy()

if SAMPLE_SIZE and len(df) > SAMPLE_SIZE:
    df = df.sample(SAMPLE_SIZE, random_state=42).reset_index(drop=True)

df.head()

,article_id,title,category,content
0,63642,Young teen wins top science prize for soap tha...,Ethiopia,"A 14-year-old boy has been named ""America's to..."
1,13370,"Vehicular homicide suspect who ""reeked of alco...",United States,"Ting Ye, 26, ""reeked of alcohol"" when she was ..."
2,69158,The Pastry Chefs Defining Restaurant Dessert R...,Guyana,Kelly Nam plates a dessert at New York City’s ...
3,97723,"Flula Borg hilarious deconstructs that weird ""...",History,"Ok, one last Halloween post and I'll abandon t..."
4,84216,Korat zoo welcomes new member - sun bear,Myanmar,Nakhon Ratchasima Zoo has a new member - a one...


In [16]:
stop_words = set(stopwords.words("english"))

def clean_text(text):
    text = str(text).lower()
    text = re.sub(r"http\S+|www\S+", " ", text)
    text = re.sub(r"[^a-z\s]", " ", text)
    tokens = text.split()
    tokens = [token for token in tokens if len(token) > 2 and token not in stop_words]
    return tokens

df["tokens"] = df[TEXT_COLUMN].apply(clean_text)
df["clean_text"] = df["tokens"].apply(lambda tokens: " ".join(tokens))

df[["title", "category", "clean_text"]].head()

,title,category,clean_text
0,Young teen wins top science prize for soap tha...,Ethiopia,year old boy named america top young scientist...
1,"Vehicular homicide suspect who ""reeked of alco...",United States,ting reeked alcohol pulled driver seat mph rol...
2,The Pastry Chefs Defining Restaurant Dessert R...,Guyana,kelly nam plates dessert new york city joomak ...
3,"Flula Borg hilarious deconstructs that weird ""...",History,one last halloween post abandon topic next yea...
4,Korat zoo welcomes new member - sun bear,Myanmar,nakhon ratchasima zoo new member one month old...


In [17]:
print(df.shape)
df["category"].fillna("Unknown").value_counts().head(10)

(5000, 6)


category
Stock          180
Technology     120
Finance        116
Real estate    115
Health         109
Canada         107
Education       97
News            93
Food            90
Jobs            86
Name: count, dtype: int64

In [18]:
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.decomposition import LatentDirichletAllocation

vectorizer = CountVectorizer(
    stop_words="english",
    max_df=0.90,
    min_df=20,
    ngram_range=(1, 2)
)

doc_term_matrix = vectorizer.fit_transform(df["clean_text"])

In [19]:
for n in [5, 8, 10, 12]:
    lda_model = LatentDirichletAllocation(
        n_components=n,
        random_state=42,
        learning_method="batch"
    )
    lda_model.fit(doc_term_matrix)
    print(f"\nTop words for {n} topics:")
    feature_names = vectorizer.get_feature_names_out()
    for topic_idx, topic in enumerate(lda_model.components_):
        top_words = [feature_names[i] for i in topic.argsort()[:-11:-1]]
        print(f"Topic {topic_idx + 1}: {', '.join(top_words)}")


Top words for 5 topics:
Topic 1: new, said, world, getty, news, images, year, time, people, south
Topic 2: report, free, free report, shares, stock, traded, research, reports, rating, price
Topic 3: nov, globe, newswire, globe newswire, nov globe, market, israel, global, new, november
Topic 4: october, year, years, like, new, million, time, november, month, life
Topic 5: quarter, free, report, free report, recent, according, quarter according, nyse, company, shares

Top words for 8 topics:
Topic 1: new, world, cup, time, apple, world cup, season, video, help, international
Topic 2: report, free, free report, research, shares, rating, stock, issued, nyse, reports
Topic 3: nov, globe, newswire, globe newswire, nov globe, market, united, global, york, new york
Topic 4: october, year, month, new, short, good, health, september, great, growth
Topic 5: quarter, free, report, free report, company, shares, november, earnings, nyse, reports
Topic 6: said, state, getty, president, images, gaza,

In [7]:
def print_lda_topics(model, feature_names, top_n=10):
    for topic_idx, topic in enumerate(model.components_):
        top_words = [feature_names[i] for i in topic.argsort()[:-top_n - 1:-1]]
        print(f"Topic {topic_idx + 1}: {', '.join(top_words)}")

feature_names = vectorizer.get_feature_names_out()
print_lda_topics(lda_model, feature_names)

Topic 1: report, free, quarter, recent, according, shares, nyse, company, second, filing
Topic 2: said, news, president, south, people, minister, city, new, images, nov
Topic 3: october, time, prime, year, apple, minister, news, short, day, month
Topic 4: report, globe, newswire, nov, research, market, free, global, rating, com
Topic 5: november, said, season, tuesday, october, series, thursday, united, time, record
Topic 6: company, million, new, earnings, announced, media, reported, quarter, years, group
Topic 7: world, israel, hamas, gaza, cup, war, new, india, including, getty
Topic 8: new, traded, state, high, week, york, low, thursday, trading, monday


In [20]:
lda_topic_matrix = lda_model.transform(doc_term_matrix)
df["lda_topic"] = lda_topic_matrix.argmax(axis=1)
df[["title", "category", "lda_topic"]].head(10)

,title,category,lda_topic
0,Young teen wins top science prize for soap tha...,Ethiopia,5
1,"Vehicular homicide suspect who ""reeked of alco...",United States,5
2,The Pastry Chefs Defining Restaurant Dessert R...,Guyana,2
3,"Flula Borg hilarious deconstructs that weird ""...",History,0
4,Korat zoo welcomes new member - sun bear,Myanmar,5
5,Striking actors say they have rejected the Hol...,Artificial Intelligence,9
6,Addus HomeCare (NASDAQ:ADUS) Rating Lowered to...,Health,1
7,India looks to fast-track approvals for Elon M...,Cars,6
8,Here Are the Favorites to Win the 2023 Nobel P...,Ukraine,6
9,Unity Software Inc. (NYSE:U) Shares Sold by Re...,Germany,7


In [16]:
dictionary = corpora.Dictionary(df["tokens"])
corpus = [dictionary.doc2bow(tokens) for tokens in df["tokens"]]
topic_terms = []

for topic in lda_model.components_:
    top_indices = topic.argsort()[:-11:-1]
    topic_terms.append([feature_names[i] for i in top_indices])

coherence_model = CoherenceModel(
    topics=topic_terms,
    texts=df["tokens"],
    dictionary=dictionary,
    coherence="c_v"
)

print({"lda_coherence": coherence_model.get_coherence()})

NameError: name 'corpora' is not defined

In [9]:
topic_model = BERTopic(language="english", calculate_probabilities=True, verbose=True)
bertopic_topics, bertopic_probs = topic_model.fit_transform(df["clean_text"].tolist())
df["bertopic_topic"] = bertopic_topics

2026-03-24 16:37:47,410 - BERTopic - Embedding - Transforming documents to embeddings.


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Batches:   0%|          | 0/157 [00:00<?, ?it/s]

2026-03-24 16:38:06,739 - BERTopic - Embedding - Completed ✓
2026-03-24 16:38:06,740 - BERTopic - Dimensionality - Fitting the dimensionality reduction algorithm
2026-03-24 16:38:30,798 - BERTopic - Dimensionality - Completed ✓
2026-03-24 16:38:30,800 - BERTopic - Cluster - Start clustering the reduced embeddings
2026-03-24 16:38:32,108 - BERTopic - Cluster - Completed ✓
2026-03-24 16:38:32,113 - BERTopic - Representation - Fine-tuning topics using representation models.
2026-03-24 16:38:32,279 - BERTopic - Representation - Completed ✓


In [10]:
topic_info = topic_model.get_topic_info()
topic_info.head(10)

,Topic,Count,Name,Representation,Representative_Docs
0,-1,1517,-1_chars_new_nov_globe,"[chars, new, nov, globe, newswire, said, marke...",[space part future inc international media gro...
1,0,488,0_quarter_recent_according_filing,"[quarter, recent, according, filing, inc, seco...",[ieq capital llc lowered position mobile inc n...
2,1,292,1_israel_gaza_hamas_israeli,"[israel, gaza, hamas, israeli, war, iran, pale...",[united nations china iran multitude arab nati...
3,2,160,2_getty_work_artificial_intelligence,"[getty, work, artificial, intelligence, htc, c...",[venturebeat presents unleashed exclusive exec...
4,3,140,3_pro_amazon_apple_samsung,"[pro, amazon, apple, samsung, iphone, deals, b...",[microsoft amazon back prime day big deal days...
5,4,126,4_film_movie_series_films,"[film, movie, series, films, show, episode, co...",[ranbir also spoke film tried find similarity ...
6,5,104,5_ukraine_russia_russian_ukrainian,"[ukraine, russia, russian, ukrainian, moscow, ...",[ukraine aim winter months cut russian militar...
7,6,100,6_crypto_bitcoin_cryptocurrency_reserve,"[crypto, bitcoin, cryptocurrency, reserve, fed...",[bitcoin cryptocurrencies advanced thursday li...
8,7,79,7_quarterback_nba_basketball_season,"[quarterback, nba, basketball, season, game, s...",[tempe ariz second time season arizona cardina...
9,8,72,8_cup_india_cricket_world,"[cup, india, cricket, world, australia, englan...",[international cricket council icc expected ea...


In [11]:
topic_model.get_topic(0)

[('quarter', np.float64(0.049808920140921656)),
 ('recent', np.float64(0.047806710485687076)),
 ('according', np.float64(0.04542645734241109)),
 ('filing', np.float64(0.04287653218780642)),
 ('inc', np.float64(0.039300233916086114)),
 ('second', np.float64(0.03669269546487431)),
 ('free', np.float64(0.035961713820082564)),
 ('nyse', np.float64(0.035516326826683216)),
 ('llc', np.float64(0.033853200976171706)),
 ('holdings', np.float64(0.033064534613084624))]

In [12]:
# Run in Jupyter to open the interactive visualisation.\n
topic_model.visualize_barchart(top_n_topics=10)

In [21]:
topic_model.visualize_topics()

In [23]:
topic_model_17 = BERTopic(nr_topics=17)
topics_17, probs_17 = topic_model_17.fit_transform(df["clean_text"].tolist())

'[WinError 10054] An existing connection was forcibly closed by the remote host' thrown while requesting HEAD https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2/resolve/main/adapter_config.json
Retrying in 1s [Retry 1/5].


RuntimeError: Cannot send a request, as the client has been closed.